# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NK0028/FlyRank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Two signal checks first** (bucket tables with n, one-word verdict each):

1. **CTR vs. position** (flag-linked — this is the signal behind FlyRank's CTR-fix logic): does a better `pos_prev30` tier associate with higher `ctr_prev30`? Bucketed below.
2. **Volume** (the signal behind quick-win logic): does higher `imp_prev30` associate with a bigger CTR shortfall against its own position tier — i.e., is there more "opportunity" sitting in high-volume pages? Bucketed below.

**The rule, in plain words:** *a page is worth reviewing if it has real search volume, sits at a position where a good CTR is realistic, but its actual CTR falls meaningfully short of what pages in that same position tier typically get.* That gap, multiplied by volume, is the score — readable on purpose, no fitted weights.

- **Score:** `baseline_action_score = imp_prev30 * max(0, tier_median_ctr - ctr_prev30)`
- **Reason code (single):** `CTR_UNDERPERFORMS_POSITION_TIER`
- **Action label (single):** `review_metadata`

This uses only `prev30` (prior-window) and query-table signals — no `imp_last30`, no `trend_direction`, no future window.

In [1]:
%pip -q install duckdb huggingface_hub

import os
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        raise RuntimeError(
            "Set HF_TOKEN as a Colab Secret (key icon, left sidebar) before running this cell."
        )

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
MONTH = "2026-03"  # mid-panel, never the _sample (final) month

# Prior-window (prev30) features only -- never last30/label-window columns.
prev30 = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
        WHERE strftime(report_date, '%Y-%m') = '{MONTH}'
    )
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions)  AS imp_prev30,
        SUM(f.gsc_clicks)       AS clk_prev30,
        AVG(f.gsc_avg_position) AS pos_prev30
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date <= b.end_d - INTERVAL 30 DAY
      AND f.report_date >  b.end_d - INTERVAL 60 DAY
    GROUP BY 1, 2
    HAVING imp_prev30 >= 50
""").df()

prev30["ctr_prev30"] = 100 * prev30["clk_prev30"] / prev30["imp_prev30"].replace(0, np.nan)
prev30["position_tier"] = pd.cut(
    prev30["pos_prev30"], bins=[-0.01, 3, 10, 20, 50, 10_000],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)

# --- Signal check 1: CTR vs. position (flag-linked -- the signal behind CTR-fix logic) ---
ctr_by_tier = prev30.groupby("position_tier", observed=True).agg(
    n=("ctr_prev30", "size"), mean_ctr=("ctr_prev30", "mean")
).reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print("Signal check 1: CTR by position tier (prev30)")
print(ctr_by_tier)

tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
ctr_values = ctr_by_tier["mean_ctr"].values
is_monotonic_decreasing = all(ctr_values[i] >= ctr_values[i+1] for i in range(len(ctr_values)-1)
                               if not (pd.isna(ctr_values[i]) or pd.isna(ctr_values[i+1])))
verdict_1 = "CONFIRMED" if is_monotonic_decreasing else "MIXED"
print(f"Verdict: {verdict_1}  (CTR should fall as position tier worsens, top_3 -> deep)\n")

# --- Signal check 2: volume vs. CTR-shortfall (the signal behind quick-win logic) ---
prev30["tier_median_ctr"] = prev30.groupby("position_tier", observed=True)["ctr_prev30"].transform("median")
prev30["ctr_shortfall"] = (prev30["tier_median_ctr"] - prev30["ctr_prev30"]).clip(lower=0)

volume_tier = pd.qcut(prev30["imp_prev30"], 4, labels=["q1_low", "q2", "q3", "q4_high"], duplicates="drop")
shortfall_by_volume = prev30.groupby(volume_tier, observed=True).agg(
    n=("ctr_shortfall", "size"), mean_ctr_shortfall=("ctr_shortfall", "mean")
)
print("Signal check 2: CTR shortfall by volume quartile (prev30)")
print(shortfall_by_volume)

shortfall_values = shortfall_by_volume["mean_ctr_shortfall"].values
is_increasing_with_volume = shortfall_values[-1] > shortfall_values[0]
verdict_2 = "CONFIRMED" if is_increasing_with_volume else "OPPOSITE"
print(f"Verdict: {verdict_2}  (higher volume should carry more CTR-shortfall 'opportunity')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal check 1: CTR by position tier (prev30)
                   n  mean_ctr
position_tier                 
top_3          11850  0.322049
page_1         47454  0.319121
striking       20589  0.235438
page_3_5       13508  0.143944
deep            1898  0.059615
Verdict: CONFIRMED  (CTR should fall as position tier worsens, top_3 -> deep)

Signal check 2: CTR shortfall by volume quartile (prev30)
                n  mean_ctr_shortfall
imp_prev30                           
q1_low      23878            0.062628
q2          23783            0.051387
q3          23814            0.032964
q4_high     23824            0.017895
Verdict: OPPOSITE  (higher volume should carry more CTR-shortfall 'opportunity')


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
qsignals = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(content_visible_query_count) AS visible_queries
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

df = prev30.merge(qsignals, on="content_hash_id", how="left")

# --- The rule: ONE score, ONE reason code, ONE action label ---
# Readable on purpose: volume x how far CTR falls short of its own position tier's median.
df["baseline_action_score"] = df["imp_prev30"] * df["ctr_shortfall"]
df["reason_code"] = "CTR_UNDERPERFORMS_POSITION_TIER"
df["action"] = "review_metadata"

df = df.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["rank", "client_hash_id", "content_hash_id", "baseline_action_score", "reason_code",
            "action", "imp_prev30", "pos_prev30", "position_tier", "ctr_prev30", "tier_median_ctr"]
df[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Scored {len(df):,} content items for month={MONTH}")
print(f"Base rate check -- score > 0 (i.e. any real shortfall at all): "
      f"{(df['baseline_action_score'] > 0).mean():.1%} of rows")
print()
print(df[out_cols].head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scored 95,299 content items for month=2026-03
Base rate check -- score > 0 (i.e. any real shortfall at all): 31.1% of rows

   rank           client_hash_id           content_hash_id  \
0     1  client_73cda7b4e4f265ea  content_9c057b66c30a3abb   
1     2  client_73cda7b4e4f265ea  content_8e1334d6356668e3   
2     3  client_73cda7b4e4f265ea  content_fec55986a1868d62   
3     4  client_73cda7b4e4f265ea  content_c9f840183215651b   
4     5  client_23a62021009f63c4  content_2ac8c7995de53cd1   
5     6  client_23a62021009f63c4  content_44f34c0a90047651   
6     7  client_e547b89c05043229  content_306bc78dff1eb683   
7     8  client_62f4a7e64f5e0096  content_15d741d8bb4143c2   
8     9  client_73cda7b4e4f265ea  content_d30a67f972196ca1   
9    10  client_3197e6291363b4db  content_22588e765b93dfac   

   baseline_action_score                      reason_code           action  \
0           35103.761755  CTR_UNDERPERFORMS_POSITION_TIER  review_metadata   
1           31832.601881  CTR_UNDERPE

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top10 = df.head(10).copy()

# One-line "what would make this wrong" per row, based on its own position tier --
# a monolithic caveat wouldn't reflect that top_3 pages and deep pages fail differently.
def caveat_for_tier(tier):
    return {
        "top_3":     "wrong if a featured snippet/rich result above it is satisfying the query without a click",
        "page_1":    "wrong if the result is a branded/navigational query where users already know they'll click through eventually regardless of snippet",
        "striking":  "wrong if this page recently moved into this tier and the CTR average hasn't caught up yet (lag, not a real problem)",
        "page_3_5":  "wrong if this is a long-tail page where low CTR is expected and the fix effort wouldn't be worth it anyway",
        "deep":      "wrong if the position itself is the real problem -- a metadata fix won't help a page nobody sees",
    }.get(str(tier), "wrong if there's context this data doesn't capture (seasonality, a recent redesign, etc.)")

for _, row in top10.iterrows():
    print(f"rank {row['rank']:>2}  score={row['baseline_action_score']:.1f}  "
          f"action={row['action']}  tier={row['position_tier']}")
    print(f"    why: CTR {row['ctr_prev30']:.2f}% is {row['tier_median_ctr'] - row['ctr_prev30']:.2f}pp "
          f"below its {row['position_tier']} tier median, on {row['imp_prev30']:.0f} prior-30d impressions")
    print(f"    what would make this wrong: {caveat_for_tier(row['position_tier'])}")


rank  1  score=35103.8  action=review_metadata  tier=page_1
    why: CTR 0.00% is 0.16pp below its page_1 tier median, on 224600 prior-30d impressions
    what would make this wrong: wrong if the result is a branded/navigational query where users already know they'll click through eventually regardless of snippet
rank  2  score=31832.6  action=review_metadata  tier=page_1
    why: CTR 0.00% is 0.16pp below its page_1 tier median, on 204368 prior-30d impressions
    what would make this wrong: wrong if the result is a branded/navigational query where users already know they'll click through eventually regardless of snippet
rank  3  score=30740.8  action=review_metadata  tier=page_1
    why: CTR 0.00% is 0.16pp below its page_1 tier median, on 196126 prior-30d impressions
    what would make this wrong: wrong if the result is a branded/navigational query where users already know they'll click through eventually regardless of snippet
rank  4  score=19866.9  action=review_metadata  tier=pa

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# --- Which top-10 picks look weakest, and why ---
# Weak = real volume is thin even though the raw score looks big (score rewards volume x gap,
# so a moderate gap on huge volume can outrank a huge gap on modest volume).
median_top10_volume = top10["imp_prev30"].median()
weak = top10[top10["imp_prev30"] < median_top10_volume]
print(f"{len(weak)} of the top 10 sit below the top-10's own median volume "
      f"({median_top10_volume:.0f} impressions) -- these are driven more by an unusually")
print("large CTR gap than by genuine scale, and deserve a second look before committing review time.")
print(weak[["rank", "imp_prev30", "ctr_prev30", "tier_median_ctr"]])
print()

# --- Leakage check: confirm no label-window or product-flag columns were used ---
feature_cols_used = {"imp_prev30", "clk_prev30", "pos_prev30", "ctr_prev30",
                     "position_tier", "tier_median_ctr", "ctr_shortfall", "visible_queries"}
forbidden_cols = {"imp_last30", "clk_last30", "pos_last30", "trend_direction", "trend_pct", "access_profile"}

leaked = feature_cols_used & forbidden_cols
assert len(leaked) == 0, f"LEAKAGE: forbidden columns used as features: {leaked}"
print("Leakage check passed: no last30/label-window column or product/access-profile flag")
print("was used anywhere in the scoring rule. Columns actually used:", sorted(feature_cols_used))


5 of the top 10 sit below the top-10's own median volume (91374 impressions) -- these are driven more by an unusually
large CTR gap than by genuine scale, and deserve a second look before committing review time.
   rank  imp_prev30  ctr_prev30  tier_median_ctr
5     6     90563.0    0.014355         0.156740
6     7     88801.0    0.061936         0.188892
7     8     86643.0    0.045012         0.156740
8     9     72177.0    0.033252         0.156740
9    10     57764.0    0.003462         0.156740

Leakage check passed: no last30/label-window column or product/access-profile flag
was used anywhere in the scoring rule. Columns actually used: ['clk_prev30', 'ctr_prev30', 'ctr_shortfall', 'imp_prev30', 'pos_prev30', 'position_tier', 'tier_median_ctr', 'visible_queries']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.